In [7]:
import torch
from transformers import BertForSequenceClassification, BertTokenizer
import re
import string
import demoji
import tkinter as tk

# Load the tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Load the best saved model
output_dir = r'C:\Users\sujal\Projects\Cyber Bullying Detection\cyberbullying_bert_model' # Directory where the best model was saved
model = BertForSequenceClassification.from_pretrained(output_dir)

# Set the model to evaluation mode
model.eval()

# Ensure everything uses the CPU device
device = torch.device("cpu")
model.to(device)

# Define the cleaning function
def initial_cleaning(text):
    """
    Cleans the input text:
    - Emoji to description
    - Lowercase
    - Remove URLs, mentions, hashtags, punctuation, excess space
    """
    text = demoji.replace_with_desc(text, sep=" ")
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'\@\w+|\#', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = text.strip()
    text = re.sub(r'\s+', ' ', text)
    return text

def predict_text(text, model, tokenizer, device, max_length=64):
    cleaned_text = initial_cleaning(text)
    encoded_dict = tokenizer.encode_plus(
        cleaned_text,
        add_special_tokens=True,
        max_length=max_length,
        padding='max_length',
        return_attention_mask=True,
        return_tensors='pt',
        truncation=True
    )
    input_ids = encoded_dict['input_ids'].to(device)
    attention_mask = encoded_dict['attention_mask'].to(device)

    with torch.no_grad():
        outputs = model(input_ids, token_type_ids=None, attention_mask=attention_mask)

    logits = outputs.logits
    probabilities = torch.softmax(logits, dim=1)
    predicted_class = torch.argmax(probabilities, dim=1).item()
    predicted_probability = probabilities[0][predicted_class].item()
    return predicted_class, predicted_probability


In [3]:

if __name__ == "__main__":
    user_input = input("Enter the text you want to classify: ")
    predicted_class, predicted_probability = predict_text(user_input, model, tokenizer, device)
    label_map = {0: "Not Cyberbullying", 1: "Cyberbullying"}
    predicted_label = label_map[predicted_class]

    print(f"\nInput Text: '{user_input}'")
    print(f"Predicted Class: {predicted_label}")
    print(f"Predicted Probability: {predicted_probability:.4f}")



Input Text: 'hey bitch'
Predicted Class: Cyberbullying
Predicted Probability: 0.9938


In [8]:


# Load model and tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertForSequenceClassification.from_pretrained(r'C:\Users\sujal\Projects\Cyber Bullying Detection\cyberbullying_bert_model')
model.eval()
model.to('cpu')

def initial_cleaning(text):
    text = demoji.replace_with_desc(text, sep=" ")
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'\@\w+|\#', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = text.strip()
    text = re.sub(r'\s+', ' ', text)
    return text

def predict_text(text, model, tokenizer, device, max_length=64):
    cleaned_text = initial_cleaning(text)
    encoded_dict = tokenizer.encode_plus(
        cleaned_text,
        add_special_tokens=True,
        max_length=max_length,
        padding='max_length',
        return_attention_mask=True,
        return_tensors='pt',
        truncation=True
    )
    input_ids = encoded_dict['input_ids'].to(device)
    attention_mask = encoded_dict['attention_mask'].to(device)
    with torch.no_grad():
        outputs = model(input_ids, token_type_ids=None, attention_mask=attention_mask)
    probabilities = torch.softmax(outputs.logits, dim=1)
    pred_class = torch.argmax(probabilities, dim=1).item()
    pred_prob = probabilities[0][pred_class].item()
    return pred_class, pred_prob

    
def on_send():
    text = input_field.get()
    pred_class, pred_prob = predict_text(text, model, tokenizer, 'cpu')
    if pred_class == 1:  # Cyberbullying detected
        warning_label.config(text="Warning! Cyberbullying detected. Message blocked.", fg="red")
    else:
        warning_label.config(text="Message sent!", fg="green")
        input_field.delete(0, tk.END)

# Simple Tkinter GUI
root = tk.Tk()
root.title("Safe Keyboard")
input_field = tk.Entry(root, width=50)
input_field.pack(pady=10)

warning_label = tk.Label(root, text="", font=("Arial", 12))
warning_label.pack(pady=5)

send_btn = tk.Button(root, text="Send", command=on_send)
send_btn.pack(pady=10)

root.mainloop()



# Example to test
RT @kohfuckyourself I'm not sexist, but Feminists make me sick in how they go about fighting for equality. Sorry.

Kat I'd love to slap your face with a pork cutlet #MKR

@jncatron @isra_jourisra @AMPalestine Islamophobia is like the idea of Naziphobia. Islam is a religion of hate and it must be outlawed.

@ummsuhaym @logicalmind11 Quran 8.12 would be a good example of terrorism. http://t.co/vonYOAtpfk